# Yahoo Finance Data Integrator - adds stock data from around each article's release date

In [15]:
# Imports
from bs4 import BeautifulSoup
import pandas as pd
import time
import dateutil
import warnings
import yfinance as yf
from datetime import datetime
from datetime import timedelta

# Supress certain warnings
from dateutil.parser import UnknownTimezoneWarning
warnings.filterwarnings("ignore", category=UnknownTimezoneWarning)

In [16]:
# Download yahoo finance daily stock data
yf_df = yf.download("NVDA", start="2000-01-01", end=datetime.today(), auto_adjust=False)

# Df reformatting for easier access
yf_df.columns = yf_df.columns.droplevel(1)
yf_df = yf_df.reset_index()

print("\nTOTAL DAYS = " + str(len(yf_df)) + "\n")
print(yf_df.head(5))
print(yf_df.tail(5))

[*********************100%***********************]  1 of 1 completed


TOTAL DAYS = 6601

Price       Date  Adj Close     Close      High       Low      Open     Volume
0     2000-01-03   0.089410  0.097526  0.099219  0.091927  0.098438  300912000
1     2000-01-04   0.087023  0.094922  0.096094  0.090104  0.095833  300480000
2     2000-01-05   0.084158  0.091797  0.093750  0.090495  0.092188  188352000
3     2000-01-06   0.078667  0.085807  0.091797  0.082292  0.091797  120480000
4     2000-01-07   0.079980  0.087240  0.088151  0.084115  0.085417   71184000
Price       Date   Adj Close       Close        High         Low        Open  \
6596  2026-03-26  171.240005  171.240005  176.509995  171.139999  176.070007   
6597  2026-03-27  167.520004  167.520004  170.970001  167.009995  170.000000   
6598  2026-03-30  165.169998  165.169998  169.449997  164.270004  168.779999   
6599  2026-03-31  174.399994  174.399994  174.619995  166.960007  166.970001   
6600  2026-04-01  175.750000  175.750000  177.369995  174.759995  176.000000   

Price     Volume  
6596  

In [17]:
# Get stock data around article publication date

def get_stock_data(time_str):
  # Get publication date
  dt = dateutil.parser.parse(time_str)

  # Don't get stock data from the same day if the article was released before market close
  if(dt.hour < 4): dt = dt - timedelta(days=1)

  # Nullify time so dt only contains date
  dt = dt.replace(hour=0, minute=0, second=0, microsecond=0)

  # Get data from publication date (search backward)
  t0_row = pd.DataFrame()
  while len(t0_row) == 0:
    t0_row = yf_df[yf_df["Date"] == dt]
    dt = dt - timedelta(days=1)

  # Get index number of data from above
  index = t0_row.iloc[0].name

  # Get data from days before and after publication date (indices = days of trading)
  df_window = yf_df.iloc[index-3:index+4].copy()

  # Only get relevant data from df_window
  tm3d = df_window.iloc[0]["Adj Close"]
  tm1d = df_window.iloc[2]["Adj Close"]
  tm0d = df_window.iloc[3]["Adj Close"]
  tp1d = df_window.iloc[4]["Adj Close"]
  tp3d = df_window.iloc[6]["Adj Close"]

  # Format as dataframe and return
  df_final = pd.DataFrame([[tm3d, tm1d, tm0d, tp1d, tp3d]],
                          columns=['T-3D', 'T-1D', 'T-0D', 'T+1D', 'T+3D'])
  return df_final

# Test
get_stock_data("Nov 06, 2025, 10:57 EST").head()

,T-3D,T-1D,T-0D,T+1D,T+3D
0,206.857285,195.188583,188.059357,188.129349,193.138794


In [18]:
# Open CSV containing article data
df = pd.DataFrame(pd.read_csv("forbes_articles_738.csv"))
dates = df['Time'].tolist()
numDates = len(dates)

df.head()

,Unnamed: 0,Link,Title,Time,Author,Body
0,0,https://www.forbes.com/sites/catherinebrock/20...,Stock Prices Finish Week Lower After Put Filin...,"Nov 07, 2025, 06:30pm EST",Catherine Brock,U.S. stocks retreated this week after investor...
1,1,https://www.forbes.com/sites/greatspeculations...,A Decade Of Rewards: $83 Bil From NVIDIA Stock,"Nov 07, 2025, 11:55am EST",Trefis Team,"Over the past ten years, NVIDIA (NVDA) stock h..."
2,2,https://www.forbes.com/sites/johnkoetsier/2025...,8 Robotics Startups Backed By Nvidia And Amazon,"Nov 06, 2025, 02:20pm EST",John Koetsier,Amazon and Nvidia are backing eight AI and rob...
3,3,https://www.forbes.com/sites/ywang/2025/11/05/...,Founder Of The ‘Nvidia Of China’ Triples His W...,"Nov 05, 2025, 04:39pm EST",Yue Wang,This story is part of Forbes’ coverage of Chin...
4,4,https://www.forbes.com/sites/jonmarkman/2025/1...,You Missed Nvidia Because You Think In Straigh...,"Nov 04, 2025, 10:57am EST",Jon Markman,Your greatest risk in this market isn’t what y...


In [19]:
data = pd.DataFrame()

for i in range(numDates):
  data = pd.concat([data, get_stock_data(dates[i])])

data = data.reset_index(drop=True)
print(len(data))
data.head()

738


,T-3D,T-1D,T-0D,T+1D,T+3D
0,198.668182,188.059357,188.129349,199.028152,193.778732
1,198.668182,188.059357,188.129349,199.028152,193.778732
2,206.857285,195.188583,188.059357,188.129349,193.138794
3,202.467773,198.668182,195.188583,188.059357,199.028152
4,202.867722,206.857285,198.668182,195.188583,188.129349


In [20]:
# Export data as CSV
df_final = pd.concat([df, data], axis=1)
df_final = df_final.drop(df_final.columns[0], axis=1)
df_final.to_csv('forbes_and_yahoo_738.csv', index=True)
df_final.head(10)

,Link,Title,Time,Author,Body,T-3D,T-1D,T-0D,T+1D,T+3D
0,https://www.forbes.com/sites/catherinebrock/20...,Stock Prices Finish Week Lower After Put Filin...,"Nov 07, 2025, 06:30pm EST",Catherine Brock,U.S. stocks retreated this week after investor...,198.668182,188.059357,188.129349,199.028152,193.778732
1,https://www.forbes.com/sites/greatspeculations...,A Decade Of Rewards: $83 Bil From NVIDIA Stock,"Nov 07, 2025, 11:55am EST",Trefis Team,"Over the past ten years, NVIDIA (NVDA) stock h...",198.668182,188.059357,188.129349,199.028152,193.778732
2,https://www.forbes.com/sites/johnkoetsier/2025...,8 Robotics Startups Backed By Nvidia And Amazon,"Nov 06, 2025, 02:20pm EST",John Koetsier,Amazon and Nvidia are backing eight AI and rob...,206.857285,195.188583,188.059357,188.129349,193.138794
3,https://www.forbes.com/sites/ywang/2025/11/05/...,Founder Of The ‘Nvidia Of China’ Triples His W...,"Nov 05, 2025, 04:39pm EST",Yue Wang,This story is part of Forbes’ coverage of Chin...,202.467773,198.668182,195.188583,188.059357,199.028152
4,https://www.forbes.com/sites/jonmarkman/2025/1...,You Missed Nvidia Because You Think In Straigh...,"Nov 04, 2025, 10:57am EST",Jon Markman,Your greatest risk in this market isn’t what y...,202.867722,206.857285,198.668182,195.188583,188.129349
5,https://www.forbes.com/sites/roomykhan/2025/11...,All Roads Lead To NVIDIA: Bankrolling Its Own ...,"Nov 03, 2025, 11:06am EST",Roomy Khan,\nNVIDIA just wrote a $5 billion check for a s...,207.017273,202.467773,206.857285,198.668182,188.059357
6,https://www.forbes.com/sites/siladityaray/2025...,Trump Says He’ll Allow China-Nvidia Deals Exce...,"Nov 03, 2025, 05:57am EST",Siladitya Ray,President Donald Trump on Sunday said he will ...,207.017273,202.467773,206.857285,198.668182,188.059357
7,https://www.forbes.com/sites/marcochiappetta/2...,"US DOE Taps Nvidia, AMD, And Oracle For Quarte...","Oct 31, 2025, 05:34pm EDT",Marco Chiappetta,"Over the last few days, the U.S. Department of...",201.007935,202.867722,202.467773,206.857285,195.188583
8,https://www.forbes.com/sites/the-prototype/202...,Nvidia Expands Into Quantum Computing And Fusi...,"Oct 31, 2025, 02:01pm EDT",Alex Knapp,"In this week’s edition of The Prototype, we lo...",201.007935,202.867722,202.467773,206.857285,195.188583
9,https://www.forbes.com/sites/carminegallo/2025...,Nvidia’s $5 Trillion Storyteller-In-Chief,"Oct 31, 2025, 05:30am EDT",Carmine Gallo,A plaque that hangs above a Denny’s booth in S...,201.007935,202.867722,202.467773,206.857285,195.188583
